# Group-Level Fiber Photometry Analysis

This notebook demonstrates how to combine processed photometry sessions across mice while preserving the mouse as the biological unit.

The analysis hierarchy is:

trials → session summary → mouse summary → group summary

This prevents mice with more trials or more sessions from automatically receiving greater weight in group-level averages.

The notebook focuses on cue-aligned photometry and related trial categories. The same framework can later be extended to solenoid, licking, locomotion, and other event types.

## 1. Configuration

Set the repository and photometry root. Then list the sessions to include.

Multiple sessions from the same mouse are allowed. They are averaged within mouse before the group mean is calculated.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\path\to\photometry-analysis"
)

PHOTOMETRY_ROOT = Path(
    r"Z:\Photometry"
)

# Sessions included in this group analysis.
# Add or remove entries as needed.
SESSIONS = [
    {"mouse": "DK21", "date": "230704", "run": 1},
    {"mouse": "DK21", "date": "230704", "run": 2},
    # {"mouse": "DK40", "date": "231005", "run": 1},
]

CHANNEL = 1

WINDOW_PRE = 5.0
WINDOW_POST = 10.0

BASELINE_START = -5.0
BASELINE_END = 0.0

print("Number of sessions:", len(SESSIONS))
for session_info in SESSIONS:
    print(session_info)

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

import src.save_sessiondata
import src.pynapple_utils

## 2. Helper functions

The functions below:

1. construct processed-session paths
2. load a processed session
3. extract event-aligned traces
4. calculate within-session averages
5. calculate mouse-level averages
6. calculate group mean and SEM

In [ ]:
def get_processed_path(
    mouse,
    date,
    run
):
    session_dir = (
        PHOTOMETRY_ROOT
        / mouse
        / f"{mouse}_{date}"
    )

    return (
        session_dir
        / f"{mouse}-{date}-{run:03d}-processed.npz"
    )


def load_processed_session(
    mouse,
    date,
    run
):
    path = get_processed_path(
        mouse,
        date,
        run
    )

    session = src.save_sessiondata.load_session(
        path
    )

    data = src.pynapple_utils.session_to_pynapple(
        session
    )

    return (
        session,
        data,
        path
    )


def event_aligned_traces(
    signal_time,
    signal,
    event_times,
    pre=5.0,
    post=10.0
):
    signal_time = np.asarray(
        signal_time,
        dtype=float
    )

    signal = np.asarray(
        signal,
        dtype=float
    )

    event_times = np.asarray(
        event_times,
        dtype=float
    )

    dt = np.median(
        np.diff(signal_time)
    )

    relative_time = np.arange(
        -pre,
        post + 0.5 * dt,
        dt
    )

    valid_indices = np.asarray(
        [
            i
            for i, event in enumerate(event_times)
            if (
                event - pre >= signal_time[0]
                and
                event + post <= signal_time[-1]
            )
        ],
        dtype=int
    )

    traces = np.full(
        (
            len(valid_indices),
            len(relative_time)
        ),
        np.nan
    )

    for row, i in enumerate(
        valid_indices
    ):

        traces[row] = np.interp(
            event_times[i] + relative_time,
            signal_time,
            signal
        )

    return (
        relative_time,
        traces,
        valid_indices
    )


def mean_trace(
    traces
):
    if len(traces) == 0:
        return np.full(
            traces.shape[1],
            np.nan
        )

    return np.nanmean(
        traces,
        axis=0
    )


def sem_trace(
    traces
):
    if len(traces) <= 1:
        return np.full(
            traces.shape[1],
            np.nan
        )

    return (
        np.nanstd(
            traces,
            axis=0,
            ddof=1
        )
        /
        np.sqrt(
            traces.shape[0]
        )
    )


def baseline_zscore_trials(
    relative_time,
    traces,
    baseline_start=-5.0,
    baseline_end=0.0
):
    baseline_mask = (
        (relative_time >= baseline_start)
        &
        (relative_time < baseline_end)
    )

    if not np.any(
        baseline_mask
    ):
        raise ValueError(
            "Baseline window contains no samples."
        )

    output = np.full_like(
        traces,
        np.nan,
        dtype=float
    )

    for i in range(
        traces.shape[0]
    ):

        baseline = traces[
            i,
            baseline_mask
        ]

        baseline_mean = np.nanmean(
            baseline
        )

        baseline_sd = np.nanstd(
            baseline,
            ddof=1
        )

        if (
            np.isfinite(baseline_sd)
            and baseline_sd > 0
        ):

            output[i] = (
                traces[i]
                - baseline_mean
            ) / baseline_sd

    return output

## 3. Process each session

For each session we calculate:

- cue-aligned dF/F trials
- cue-lick trial mean
- miss trial mean
- trial-local z-scored versions
- trial counts

The session is summarized before sessions are combined across mice.

In [ ]:
session_results = []

for info in SESSIONS:

    mouse = info["mouse"]
    date = info["date"]
    run = info["run"]

    print(
        f"Processing {mouse} {date} run {run}..."
    )

    session, data, path = load_processed_session(
        mouse,
        date,
        run
    )

    cue_onset = np.asarray(
        session["cue_onset"],
        dtype=float
    )

    cue_lick = np.asarray(
        session["cue_lick"],
        dtype=bool
    )

    cue_miss = np.asarray(
        session["cue_miss"],
        dtype=bool
    )

    dff_time = np.asarray(
        data[f"dff_ch{CHANNEL}"].t,
        dtype=float
    )

    dff = np.asarray(
        data[f"dff_ch{CHANNEL}"].d,
        dtype=float
    )

    (
        relative_time,
        cue_dff,
        valid_indices
    ) = event_aligned_traces(
        dff_time,
        dff,
        cue_onset,
        pre=WINDOW_PRE,
        post=WINDOW_POST
    )

    valid_cue_lick = cue_lick[
        valid_indices
    ]

    valid_cue_miss = cue_miss[
        valid_indices
    ]

    cue_lick_traces = cue_dff[
        valid_cue_lick
    ]

    miss_traces = cue_dff[
        valid_cue_miss
    ]

    cue_lick_mean = mean_trace(
        cue_lick_traces
    )

    miss_mean = mean_trace(
        miss_traces
    )

    # Trial-local dF/F z-score
    cue_dff_z = baseline_zscore_trials(
        relative_time,
        cue_dff,
        baseline_start=BASELINE_START,
        baseline_end=BASELINE_END
    )

    cue_lick_z = cue_dff_z[
        valid_cue_lick
    ]

    miss_z = cue_dff_z[
        valid_cue_miss
    ]

    session_results.append(
        {
            "mouse": mouse,
            "date": date,
            "run": run,
            "path": path,
            "relative_time": relative_time,
            "cue_lick_mean": cue_lick_mean,
            "miss_mean": miss_mean,
            "cue_lick_z_mean": mean_trace(
                cue_lick_z
            ),
            "miss_z_mean": mean_trace(
                miss_z
            ),
            "n_cue_lick": len(
                cue_lick_traces
            ),
            "n_miss": len(
                miss_traces
            ),
            "n_total_valid": len(
                cue_dff
            )
        }
    )

    print(
        f"  cue lick n = {len(cue_lick_traces)}"
    )

    print(
        f"  miss n = {len(miss_traces)}"
    )

## 4. Inspect session-level results

Each session has now been reduced to a mean trace.

No mouse has been weighted by its number of trials at the group level.

In [ ]:
print(
    "Session summaries:"
)

for result in session_results:

    print(
        f"{result['mouse']} "
        f"{result['date']} "
        f"run {result['run']}: "
        f"cue lick n={result['n_cue_lick']}, "
        f"miss n={result['n_miss']}"
    )

## 5. Combine sessions within mouse

If a mouse contributed more than one session, those session-level traces are averaged first.

Thus:

```text
Mouse DK21
  run 1 ─┐
  run 2 ─┴─→ DK21 mouse mean
```

rather than allowing DK21 to receive twice the weight of another mouse simply because it contributed two sessions.

In [ ]:
mouse_groups = {}

for result in session_results:

    mouse = result["mouse"]

    if mouse not in mouse_groups:
        mouse_groups[mouse] = []

    mouse_groups[mouse].append(
        result
    )


mouse_results = {}

for mouse, results in mouse_groups.items():

    cue_lick_stack = np.vstack(
        [
            result["cue_lick_mean"]
            for result in results
        ]
    )

    miss_stack = np.vstack(
        [
            result["miss_mean"]
            for result in results
        ]
    )

    cue_lick_z_stack = np.vstack(
        [
            result["cue_lick_z_mean"]
            for result in results
        ]
    )

    miss_z_stack = np.vstack(
        [
            result["miss_z_mean"]
            for result in results
        ]
    )

    mouse_results[mouse] = {
        "relative_time": results[0]["relative_time"],

        "cue_lick_mean": np.nanmean(
            cue_lick_stack,
            axis=0
        ),

        "miss_mean": np.nanmean(
            miss_stack,
            axis=0
        ),

        "cue_lick_z_mean": np.nanmean(
            cue_lick_z_stack,
            axis=0
        ),

        "miss_z_mean": np.nanmean(
            miss_z_stack,
            axis=0
        ),

        "n_sessions": len(results)
    }

print(
    "Mouse summaries:"
)

for mouse, result in mouse_results.items():

    print(
        mouse,
        "sessions =",
        result["n_sessions"]
    )

## 6. Group mean and SEM

The group mean and SEM are now calculated across **mouse-level traces**.

Therefore, each mouse contributes one observation to the group summary.

In [ ]:
mouse_names = list(
    mouse_results.keys()
)

cue_lick_mouse_matrix = np.vstack(
    [
        mouse_results[mouse]["cue_lick_mean"]
        for mouse in mouse_names
    ]
)

miss_mouse_matrix = np.vstack(
    [
        mouse_results[mouse]["miss_mean"]
        for mouse in mouse_names
    ]
)

cue_lick_z_mouse_matrix = np.vstack(
    [
        mouse_results[mouse]["cue_lick_z_mean"]
        for mouse in mouse_names
    ]
)

miss_z_mouse_matrix = np.vstack(
    [
        mouse_results[mouse]["miss_z_mean"]
        for mouse in mouse_names
    ]
)

group_time = mouse_results[
    mouse_names[0]
]["relative_time"]

group_cue_lick_mean = np.nanmean(
    cue_lick_mouse_matrix,
    axis=0
)

group_miss_mean = np.nanmean(
    miss_mouse_matrix,
    axis=0
)

group_cue_lick_sem = sem_trace(
    cue_lick_mouse_matrix
)

group_miss_sem = sem_trace(
    miss_mouse_matrix
)

group_cue_lick_z_mean = np.nanmean(
    cue_lick_z_mouse_matrix,
    axis=0
)

group_miss_z_mean = np.nanmean(
    miss_z_mouse_matrix,
    axis=0
)

group_cue_lick_z_sem = sem_trace(
    cue_lick_z_mouse_matrix
)

group_miss_z_sem = sem_trace(
    miss_z_mouse_matrix
)

print(
    "Number of mice:",
    len(mouse_names)
)

print(
    "Mice:",
    mouse_names
)

## 7. Group raw dF/F response

Thin traces show individual mouse means. The thick traces show the group mean and the shaded region shows SEM across mice.

In [ ]:
plt.figure(
    figsize=(10, 6)
)

for row, mouse in enumerate(
    mouse_names
):

    plt.plot(
        group_time,
        cue_lick_mouse_matrix[row],
        alpha=0.35
    )

    plt.plot(
        group_time,
        miss_mouse_matrix[row],
        alpha=0.20,
        linestyle="--"
    )

plt.plot(
    group_time,
    group_cue_lick_mean,
    linewidth=3,
    label="Cue lick group mean"
)

plt.fill_between(
    group_time,
    group_cue_lick_mean - group_cue_lick_sem,
    group_cue_lick_mean + group_cue_lick_sem,
    alpha=0.2
)

plt.plot(
    group_time,
    group_miss_mean,
    linewidth=3,
    linestyle="--",
    label="Miss group mean"
)

plt.fill_between(
    group_time,
    group_miss_mean - group_miss_sem,
    group_miss_mean + group_miss_sem,
    alpha=0.2
)

plt.axvline(
    0,
    linestyle="--"
)

plt.axhline(
    0,
    linestyle=":"
)

plt.xlabel(
    "Time from cue onset (s)"
)

plt.ylabel(
    "dF/F"
)

plt.title(
    f"Group cue response ({len(mouse_names)} mice)"
)

plt.legend()

plt.tight_layout()
plt.show()

## 8. Group trial-baseline z-score

The same comparison is shown after within-trial baseline normalization.

The baseline window is configured at the top of the notebook.

In [ ]:
plt.figure(
    figsize=(10, 6)
)

for row in range(
    cue_lick_z_mouse_matrix.shape[0]
):

    plt.plot(
        group_time,
        cue_lick_z_mouse_matrix[row],
        alpha=0.25
    )

    plt.plot(
        group_time,
        miss_z_mouse_matrix[row],
        alpha=0.15,
        linestyle="--"
    )

plt.plot(
    group_time,
    group_cue_lick_z_mean,
    linewidth=3,
    label="Cue lick group mean"
)

plt.fill_between(
    group_time,
    group_cue_lick_z_mean - group_cue_lick_z_sem,
    group_cue_lick_z_mean + group_cue_lick_z_sem,
    alpha=0.2
)

plt.plot(
    group_time,
    group_miss_z_mean,
    linewidth=3,
    linestyle="--",
    label="Miss group mean"
)

plt.fill_between(
    group_time,
    group_miss_z_mean - group_miss_z_sem,
    group_miss_z_mean + group_miss_z_sem,
    alpha=0.2
)

plt.axvline(
    0,
    linestyle="--"
)

plt.axhline(
    0,
    linestyle=":"
)

plt.xlabel(
    "Time from cue onset (s)"
)

plt.ylabel(
    "Trial-baseline z-score"
)

plt.title(
    f"Group cue response: trial-local z-score ({len(mouse_names)} mice)"
)

plt.legend()

plt.tight_layout()
plt.show()

## 9. Export mouse-level results

The dictionaries below keep the mouse-level traces available for later statistical analysis.

This is preferable to saving only the final group mean because mouse-level variability is needed for inferential statistics.

In [ ]:
group_results = {
    "mouse_names": mouse_names,
    "time": group_time,

    "cue_lick_mouse_matrix": (
        cue_lick_mouse_matrix
    ),

    "miss_mouse_matrix": (
        miss_mouse_matrix
    ),

    "cue_lick_z_mouse_matrix": (
        cue_lick_z_mouse_matrix
    ),

    "miss_z_mouse_matrix": (
        miss_z_mouse_matrix
    ),

    "cue_lick_group_mean": (
        group_cue_lick_mean
    ),

    "cue_lick_group_sem": (
        group_cue_lick_sem
    ),

    "miss_group_mean": (
        group_miss_mean
    ),

    "miss_group_sem": (
        group_miss_sem
    ),

    "cue_lick_z_group_mean": (
        group_cue_lick_z_mean
    ),

    "cue_lick_z_group_sem": (
        group_cue_lick_z_sem
    ),

    "miss_z_group_mean": (
        group_miss_z_mean
    ),

    "miss_z_group_sem": (
        group_miss_z_sem
    )
}

print(
    "Group results prepared."
)

## 10. Interpretation

This notebook separates the levels of analysis:

```text
individual trials
      ↓
session mean
      ↓
mouse mean
      ↓
group mean ± SEM
```

The group SEM reflects variability across mice, not variability across individual trials.

The next step can add formal mouse-level statistics, additional event types, and group comparisons.